# NCRB scanned tables: cloud OCR, round 3

Reads **91 files (512 pages)**: the 50 ADSI suicide and accident tables of 1994–2000 recovered from the Internet Archive's copies of NCRB's old site (State-wise causes, means, profession, education, incidence and rate), and the 1974–1985 means and causes appendices whose first readings did not add up. Two readings of every page at 2048 px: OvisOCR2 and PaddleOCR-VL-1.6. Colab downloads every file itself (from web.archive.org and ncrb.gov.in); nothing is uploaded.

1. **Runtime → Change runtime type → A100**.
2. Run cells 1–4. Cell 2 asks once for Drive access. Cell 3 starts the job detached. Cell 4 shows progress and releases the GPU at the last page (about 20–30 minutes, roughly 3 compute units).
3. Afterwards, on drive.google.com right-click `ocrkit/ncrb-round3` → Download, and tell Claude where the zip is.

## 1. Settings

In [ ]:
PROJECT  = 'ncrb-round3'
MANIFEST = 'https://raw.githubusercontent.com/reclaimchennai/NCRB/main/colab/plans/round3/manifest.json'
BAKEOFF  = None
RUNS     = [{'model': 'ATH-MaaS/OvisOCR2', 'edge': 2048}, {'model': 'PaddlePaddle/PaddleOCR-VL-1.6', 'edge': 2048}]
FILES    = None
HOURS    = 3


## 2. Drive, ocrkit and vLLM (3–5 minutes)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q -U vllm "ocrkit @ git+https://github.com/reclaimchennai/NCRB#subdirectory=ocrkit" 2>&1 | tail -2
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
import json, subprocess
cap = float(subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], capture_output=True, text=True).stdout.split()[0])
import os
OUT = f'/content/drive/MyDrive/ocrkit/{PROJECT}'
os.makedirs(OUT, exist_ok=True)          # the job's log goes next to it
plan = {'name': PROJECT, 'manifest': MANIFEST, 'out': OUT, 'runs': RUNS, 'hours': HOURS, 'dtype': 'auto' if cap >= 8 else 'half'}
if BAKEOFF: plan['bakeoff'] = BAKEOFF
if FILES: plan['files'] = FILES
open('/content/plan.json', 'w').write(json.dumps(plan, indent=1))
print(json.dumps(plan, indent=1))

## 3. Start the job (detached)

In [ ]:
import subprocess, time
pids = lambda: subprocess.run(['pgrep', '-f', '[o]crkit.job'], capture_output=True, text=True).stdout.split()
if pids(): print('already running', pids())
else:
    subprocess.Popen(['python', '-m', 'ocrkit.job', '/content/plan.json'], stdout=open(f'/content/drive/MyDrive/ocrkit/{PROJECT}.log', 'a'),
                     stderr=subprocess.STDOUT, start_new_session=True)
    time.sleep(20); print('started', pids())

## 4. Progress (safe to stop and re-run; releases the GPU at the end)

In [ ]:
from ocrkit.monitor import watch
watch(OUT)

## If something looks wrong

Paste the output of this cell (it only reads):

In [ ]:
print(open(f'{OUT}/status.json').read()[-2500:])
!pgrep -af "[o]crkit" | cut -c1-150; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
!tail -15 /content/drive/MyDrive/ocrkit/{PROJECT}.log